In [ ]:
# Multi-Metric Evaluation Framework for LLM Responses in Healthcare
# Authors: Macdonald Emeshieobi

import os
import re
import json
import warnings
from typing import List, Optional
from xml.etree import ElementTree as ET

import numpy as np
import pandas as pd
from numpy.linalg import norm

from dotenv import load_dotenv

import requests
from Bio import Entrez
from bs4 import BeautifulSoup

from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.schema import Document

from rank_bm25 import BM25Okapi

import openai
from openai import OpenAI
from anthropic import Anthropic

warnings.simplefilter(action='ignore', category=FutureWarning)

load_dotenv()
OPENAI_KEY   = os.getenv("OPENAI_API_KEY")
CLAUDE_KEY   = os.getenv("ANTHROPIC_API_KEY")
DEEPSEEK_KEY = os.getenv("DEEPSEEK_API_KEY")

Entrez.email = "emeshieobim@gmail.com"

print("Imports loaded")
print(f"OpenAI key:   {'loaded' if OPENAI_KEY else 'MISSING'}")
print(f"Claude key:   {'loaded' if CLAUDE_KEY else 'MISSING'}")
print(f"DeepSeek key: {'loaded' if DEEPSEEK_KEY else 'MISSING'}")

In [ ]:
sample_queries = pd.read_csv("sample_responses.csv")
sample_queries = sample_queries.head(32)

print(f"Loaded {len(sample_queries)} queries")
print(f"Columns: {list(sample_queries.columns)}")
sample_queries.head()

In [ ]:
def get_medical_advice_chatgpt(patient_query: str, api_key: str, model: str = "gpt-3.5-turbo") -> Optional[str]:
    """Generate a medical response using ChatGPT."""
    if not patient_query or not isinstance(patient_query, str):
        raise ValueError("Patient query must be a non-empty string")

    system_message = """You are a highly knowledgeable and compassionate doctor.
1. Carefully analyze the patient's symptoms and concerns.
2. Suggest possible causes, but never give a definitive diagnosis.
3. Offer evidence-based advice for relief and care.
4. Always recommend in-person consultation with a healthcare provider.
5. Be clear, empathetic, and professional in tone.
6. Consider lifestyle and other details mentioned by the patient.
7. Avoid exaggeration or alarming language."""

    try:
        client = OpenAI(api_key=api_key)
        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": system_message},
                {"role": "user",   "content": patient_query}
            ],
            temperature=0.2,
            max_tokens=800
        )
        return response.choices[0].message.content
    except Exception as e:
        print(f"ChatGPT error: {e}")
        return None


def get_medical_advice_claude(patient_query: str, api_key: str, model: str = "claude-3-opus-20240229") -> Optional[str]:
    """Generate a medical response using Claude."""
    if not patient_query or not isinstance(patient_query, str):
        raise ValueError("Patient query must be a non-empty string")

    system_message = """You are a highly knowledgeable and compassionate doctor.
1. Carefully analyze the patient's symptoms and concerns.
2. Suggest possible causes, but never give a definitive diagnosis.
3. Offer evidence-based advice for relief and care.
4. Always recommend in-person consultation with a healthcare provider.
5. Be clear, empathetic, and professional in tone.
6. Consider lifestyle and other details mentioned by the patient.
7. Avoid exaggeration or alarming language."""

    try:
        client = Anthropic(api_key=api_key)
        response = client.messages.create(
            model=model,
            system=system_message,
            max_tokens=800,
            temperature=0.2,
            messages=[{"role": "user", "content": patient_query}]
        )
        return response.content[0].text
    except Exception as e:
        print(f"Claude error: {e}")
        return None


def get_medical_advice_deepseek(patient_query: str, api_key: str, model: str = "deepseek-chat") -> Optional[str]:
    """Generate a medical response using DeepSeek V3."""
    if not patient_query or not isinstance(patient_query, str):
        raise ValueError("Patient query must be a non-empty string")

    system_message = """You are a highly knowledgeable and compassionate doctor.
1. Carefully analyze the patient's symptoms and concerns.
2. Suggest possible causes, but never give a definitive diagnosis.
3. Offer evidence-based advice for relief and care.
4. Always recommend in-person consultation with a healthcare provider.
5. Be clear, empathetic, and professional in tone.
6. Consider lifestyle and other details mentioned by the patient.
7. Avoid exaggeration or alarming language."""

    try:
        url = "https://api.deepseek.com/v1/chat/completions"
        headers = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"}
        payload = {
            "model": model,
            "messages": [
                {"role": "system", "content": system_message},
                {"role": "user",   "content": patient_query}
            ],
            "temperature": 0.2,
            "max_tokens": 800
        }
        response = requests.post(url, headers=headers, json=payload)
        if response.status_code == 200:
            return response.json()["choices"][0]["message"]["content"].strip()
        else:
            print(f"DeepSeek API error {response.status_code}: {response.text}")
            return None
    except Exception as e:
        print(f"DeepSeek error: {e}")
        return None


print("LLM response functions defined (ChatGPT, Claude, DeepSeek)")

In [ ]:
def generate_pubmed_query(natural_query: str, api_key: str) -> str:
    """Transform a natural language medical query into a structured PubMed search string using GPT-4o."""
    system_prompt = """You are a PubMed search expert that creates high-recall queries.
Use MeSH terms and synonyms. Format output with triple quotes (''') at start and end.
Structure: (concept1 OR synonym) AND (concept2 OR synonym)
Output ONLY the search string in triple quotes, nothing else."""

    user_prompt = f"""Convert this patient query to a ready-to-use PubMed search string:

Example:
Input: "chest pain and shortness of breath at night"
Output:
'''
("chest pain" OR "angina" OR "thoracic pain")
AND
("shortness of breath" OR "dyspnoea" OR "nocturnal dyspnea")
'''

Now convert: {natural_query}"""

    try:
        client = OpenAI(api_key=api_key)
        response = client.chat.completions.create(
            model="gpt-4o",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user",   "content": user_prompt}
            ],
            temperature=0.1,
            max_tokens=150
        )
        raw = response.choices[0].message.content.strip()
        if not raw.startswith("'''") or not raw.endswith("'''"):
            raw = f"'''\n{raw}\n'''"
        return raw
    except Exception as e:
        print(f"Query transform error: {e}")
        terms = re.findall(r'\b[\w-]+\b', natural_query.lower())[:2]
        return f"'''\n\"{terms[0]}\" AND \"{terms[1]}\"\n'''" if len(terms) > 1 else f"'''\n\"{terms[0]}\"\n'''"


def fetch_pubmed_docs(query: str, retmax: int = 50) -> List[dict]:
    """Fetch abstracts from PubMed for a given search query."""
    try:
        search_handle = Entrez.esearch(db="pubmed", term=query, retmax=retmax)
        search_results = Entrez.read(search_handle)
        search_handle.close()
        id_list = search_results["IdList"]
        if not id_list:
            return []

        fetch_handle = Entrez.efetch(db="pubmed", id=",".join(id_list), rettype="abstract", retmode="xml")
        xml_data = fetch_handle.read()
        fetch_handle.close()

        root = ET.fromstring(xml_data)
        results = []
        for article in root.findall(".//PubmedArticle"):
            pmid     = article.findtext(".//PMID", default="").strip()
            title    = article.findtext(".//ArticleTitle", default="").strip()
            abstract = " ".join(
                elem.text.strip()
                for elem in article.findall(".//Abstract/AbstractText")
                if elem.text
            )
            date_elem = article.find(".//PubDate")
            year = date_elem.findtext("Year", default="Unknown") if date_elem is not None else "Unknown"
            results.append({"pmid": pmid, "title": title, "abstract": abstract, "year": year})
        return results
    except Exception as e:
        print(f"PubMed fetch error: {e}")
        return []


def fetch_europepmc_docs(query: str, page_size: int = 50) -> List[dict]:
    """Fetch abstracts and conclusions from EuropePMC."""
    url = "https://www.ebi.ac.uk/europepmc/webservices/rest/search"
    params = {"query": query + " OPEN_ACCESS:Y", "format": "json", "pageSize": page_size}
    documents = []
    try:
        response = requests.get(url, params=params)
        data = response.json()
        for res in data.get('resultList', {}).get('result', []):
            doc = {
                "title":      res.get('title'),
                "year":       res.get('pubYear'),
                "pmid":       res.get('pmcid', None),
                "abstract":   None,
                "conclusion": None
            }
            pmcid = res.get('pmcid')
            if pmcid:
                ft_url = f"https://www.ebi.ac.uk/europepmc/webservices/rest/{pmcid}/fullTextXML"
                ft_response = requests.get(ft_url)
                if ft_response.status_code == 200:
                    soup = BeautifulSoup(ft_response.content, 'xml')
                    abstract_tag = soup.find('abstract')
                    if abstract_tag:
                        doc['abstract'] = abstract_tag.get_text(separator=' ', strip=True)
                    for sec in soup.find_all('sec'):
                        title_tag = sec.find('title')
                        if title_tag and 'conclusion' in title_tag.get_text(strip=True).lower():
                            doc['conclusion'] = sec.get_text(separator=' ', strip=True)
                            break
            documents.append(doc)
    except Exception as e:
        print(f"EuropePMC fetch error: {e}")
    return documents


print("Query transformer and retrieval functions defined (PubMed + EuropePMC)")

In [ ]:
def normalize_scores(scores: List[float]) -> List[float]:
    """Min-max normalise a list of scores to [0, 1]."""
    min_s, max_s = min(scores), max(scores)
    if max_s == min_s:
        return [1.0] * len(scores)
    return [(s - min_s) / (max_s - min_s) for s in scores]


def bm25_score(query_tokens: List[str], docs: List[Document]) -> List[float]:
    """Calculate normalised BM25 scores for a query against a list of documents."""
    tokenized_corpus = [doc.page_content.split() for doc in docs]
    bm25 = BM25Okapi(tokenized_corpus)
    return normalize_scores(bm25.get_scores(query_tokens))


def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    """Cosine similarity between two vectors."""
    return np.dot(a, b) / (norm(a) * norm(b))


def dense_score(query: str, docs: List[Document], embeddings: OpenAIEmbeddings) -> List[float]:
    """Calculate normalised dense cosine similarity scores using OpenAI embeddings."""
    query_emb = embeddings.embed_query(query)
    doc_embs  = [embeddings.embed_query(doc.page_content) for doc in docs]
    scores    = [cosine_similarity(query_emb, emb) for emb in doc_embs]
    return normalize_scores(scores)


def build_document_dataframe(pub_docs: List[dict], epmc_docs: List[dict]) -> Optional[pd.DataFrame]:
    """Combine PubMed and EuropePMC documents into a single cleaned DataFrame."""
    df1 = pd.DataFrame(pub_docs)
    df2 = pd.DataFrame(epmc_docs)

    if not df1.empty:
        df1['text'] = df1['abstract']
    if not df2.empty:
        df2['text'] = df2['abstract'].fillna('').astype(str) + ' ' + df2['conclusion'].fillna('').astype(str)

    df = pd.concat([df1, df2], ignore_index=True)
    if df.empty:
        return None

    df = df.apply(lambda col: col.map(lambda x: x.strip() if isinstance(x, str) else x))
    df.replace('', np.nan, inplace=True)
    df = df.drop(columns=[c for c in ['conclusion', 'abstract'] if c in df.columns])
    df = df.dropna(subset=['text'])
    return df if not df.empty else None


def generate_pseudo_gold(query: str, top_docs: List[str], api_key: str) -> Optional[str]:
    """Synthesise a pseudo-gold reference document from the top retrieved documents using GPT-4o."""
    context = "\n\n".join([f"Document {i+1}:\n{doc.strip()}" for i, doc in enumerate(top_docs) if doc.strip()])
    prompt = f"""You are a factual research assistant.
Given these biomedical research documents and a patient query, create a concise pseudo-gold summary.
Only use facts explicitly stated in the documents. Use inline citations [Doc N].
Do not invent or hallucinate any facts.

Query: {query}

Documents:
{context}

Write a factually grounded summary that directly addresses the query."""

    try:
        client = OpenAI(api_key=api_key)
        response = client.chat.completions.create(
            model="gpt-4o",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.1,
            max_tokens=2800
        )
        return response.choices[0].message.content
    except Exception as e:
        print(f"Pseudo-gold generation error: {e}")
        return None


print("Hybrid ranking and pseudo-gold synthesis functions defined")

In [ ]:
def evaluate_response(query: str, llm_response: str, retrieved_docs: List[str], api_key: str, model: str = "gpt-4o") -> str:
    """Evaluate an LLM response across 5 metrics using GPT-4o as judge."""
    context = "\n\n".join([f"Document {i+1}:\n{doc.strip()}" for i, doc in enumerate(retrieved_docs)])
    eval_prompt = f"""You are an expert medical QA evaluator.
Evaluate the LLM response on these criteria (score each 1-10):
1. Correctness   — Are statements medically accurate based on the retrieved documents?
2. Hallucination — Does the answer contain unsupported or fabricated facts? (10 = no hallucination)
3. Completeness  — Does it fully address the patient's question?
4. Faithfulness  — Is the response faithful to the retrieved documents?
5. Groundedness  — Are claims supported with evidence from the documents?

Also note the PMIDs and publication years of the most relevant documents used.
End your response with scores on their own line: [correctness, hallucination, completeness, faithfulness, groundedness]

---
Patient Query: {query}
LLM Response:  {llm_response}
Retrieved Documents:
{context}"""

    try:
        client = OpenAI(api_key=api_key)
        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": "You are a clinical QA evaluation expert."},
                {"role": "user",   "content": eval_prompt}
            ],
            temperature=0.2,
            max_tokens=2000
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        print(f"Evaluation error: {e}")
        return "Error during evaluation."


def evaluate_empathy(question: str, chatgpt_r: str, claude_r: str, deepseek_r: str, api_key: str) -> dict:
    """Evaluate empathy scores for all three model responses using GPT-4o."""
    prompt = f"""You are an empathy evaluator. Rate each response 1-10.
Empathy requires: acknowledging emotions, validating concerns, and offering support.

Examples:
"Numbness happens. Wait it out." → 1/10 (dismissive)
"I hear how distressing this is. Would talking about it help?" → 8/10 (strong validation)
"That sounds terrifying. Let's review your symptoms together." → 10/10 (emotionally attuned)

Evaluate these responses to:
Q: {question}
A1 (ChatGPT):  {chatgpt_r}
A2 (Claude):   {claude_r}
A3 (DeepSeek): {deepseek_r}

Return valid JSON only:
{{
    "ratings": [
        {{"model": "ChatGPT",  "rating": 0, "explanation": "brief reason"}},
        {{"model": "Claude",   "rating": 0, "explanation": "brief reason"}},
        {{"model": "DeepSeek", "rating": 0, "explanation": "brief reason"}}
    ],
    "summary": "overall comparison"
}}"""

    try:
        client = OpenAI(api_key=api_key)
        response = client.chat.completions.create(
            model="gpt-4o",
            messages=[
                {"role": "system", "content": "You are a precise empathy evaluator. Always return valid JSON."},
                {"role": "user",   "content": prompt}
            ],
            temperature=0.1,
            max_tokens=800
        )
        return json.loads(response.choices[0].message.content)
    except Exception as e:
        return {"error": str(e)}


print("GPT-4o judge and empathy evaluator defined")